### Data Reading

In [0]:
df = spark.read.format("csv") \
    .option("inferSchema", True) \
    .option("header", True) \
    .load("/Volumes/workspace/default/practice/BigMart Sales.csv")

In [0]:
df.display()

### Data Reading From JSON

In [0]:
df_json = spark.read.format('json').option('inferSchema', True).option('header', True).load("/Volumes/workspace/default/practice/drivers.json")

In [0]:
df_json.display()

### Schema Defination

In [0]:
df.printSchema()


## Transformations

### Select

In [0]:
df.select('Item_Identifier','Item_Type','Item_MRP').display()

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

### Alias

In [0]:
# Create alias for a column
df_alias = df.select(col("Item_Identifier").alias("Item_ID"))

# Display in Databricks
df_alias.display()

###Filter/Where

####Filter the data with fat content = Regular

In [0]:
df_filter = df.filter(col("Item_Fat_Content")=="Regular")
display(df_filter)

#### Slice the data with item type = Soft Drinks and weight < 10

In [0]:
df_filter2 = df.filter(
    (col("Item_Type") == "Soft Drinks") & (col("Item_Weight").cast("double") < 10)
)
display(df_filter2)


#### Fetch the data with Tier in (Tier 1 or Tier 2) and outlet Size is NULL

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

df_filter3 = df.filter(
    (col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))
)

display(df_filter3)

####withColumnRenamed

In [0]:
# Rename one column
df_renamed = df.withColumnRenamed("Item_Identifier", "Item_ID")
df_renamed.printSchema()

#### withColumn : In PySpark, withColumn is a method used to add a new column or replace an existing one in a DataFrame. It’s very powerful because you can apply transformations, expressions, or functions to create new data fields.

In [0]:
from pyspark.sql.functions import when, col

# Create a new DataFrame by adding/replacing the column "Item_Price_Double"
df_clean = df.withColumn(
    "Item_Price_Double",  # Name of the new column

    # Use 'when' to apply conditional logic:
    # If Item_MRP matches a numeric pattern (integer or decimal), cast it to double and multiply by 2
    when(
        col("Item_MRP").rlike("^[0-9]+(\\.[0-9]+)?$"),  # Regex: only digits, optional decimal part
        col("Item_MRP").cast("double") * 2              # Safe numeric conversion + multiplication
    )
    # Otherwise (if Item_MRP is not numeric), set the value to NULL
    .otherwise(None)
)

# Display the cleaned DataFrame (Databricks-specific function)
display(df_clean)


#### Scenario - 2, For Using literal values that adds a new column with constant value

In [0]:
df_literal = df.withColumn('flag',lit("new")) 
display(df_literal)

#### Scenario - 3, Updating Column item

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 
df_UpdateColumn = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

display(df_UpdateColumn)

#### Sort : sorting is done using either orderBy() or sort(). Both functions are interchangeable — they return a new DataFrame with rows ordered according to the specified columns.

#### Scenario 1 : Shorting in Ascending & Descending

In [0]:
from pyspark.sql.functions import col

df_sorted = df.sort(col("Item_MRP").asc()) # same for descinding order we use. desc()
#df_sorted = df_sorted.limit(10)
display(df_sorted)

#### Scenario 2 : Sorting based on multiple column

In [0]:
df_shorted2 = df.sort(['Item_Weight','Item_Visibility'],ascending = [0,0])
df_shorted2 = df_shorted2.limit(10)
display(df_shorted2)

#### Drop Method : In PySpark, the drop method is used to remove one or more columns from a DataFrame. It’s straightforward and very useful when cleaning up your dataset.

In [0]:
df_new = df.drop("Item_Visibility")
df_new.printSchema()


In [0]:
# Droping multiple column
df_new2 = df.drop("Item_Visibility", "Outlet_Size", "Outlet_Type")
df_new2.printSchema()


#### Drop Duplicates

In [0]:
df_dup1 = df.dropDuplicates()
display(df_dup1)

In [0]:
# Want to drop duplicates based on a specific column (like Item_Type):
# This keeps only one row per unique Item_Type.

df_no_dupes = df.dropDuplicates(["Item_Type"])
display(df_no_dupes)


### Advance Transformations

#### Union : Union is used to combine two DataFrames
- ###### Combines two DataFrames row‑wise.
- ##### Requires both DataFrames to have the same schema (same number of columns, same order, same data types).
- ###### If column order differs, will get mismatched data.

In [0]:
# First DataFrame
data1 = [("A101", 5.2, "Low Fat"),
         ("A102", 8.5, "Regular")]
df1 = spark.createDataFrame(data1, ["Item_Identifier", "Item_Weight", "Item_Fat_Content"])

# Second DataFrame
data2 = [("A103", 12.0, "Low Fat"),
         ("A104", 7.8, "Regular")]
df2 = spark.createDataFrame(data2, ["Item_Identifier", "Item_Weight", "Item_Fat_Content"])

# Union (schemas must match)
df_union = df1.union(df2)

df_union.display()

#### Now what if we change order of column and try to union

In [0]:
# First DataFrame
data1 = [("A101", 5.2, "Low Fat"),
         ("A102", 8.5, "Regular")]
schema1 = 'id String, wieght double, fat String' 
df1 = spark.createDataFrame(data1, schema1)

# Second DataFrame
data2 = [(12.0, 'A103', "Low Fat"),
         (13.7,'A104', "Regular")]
schema2 = 'weight double, id String, fat String'
df2 = spark.createDataFrame(data2,schema2)

# Union (schemas must match)
df_union = df1.union(df2)

df_union.display()

#[CAST_INVALID_INPUT] The value 'A101' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018 throw error

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8019388252412816>, line 16
     13 # Union (schemas must match)
     14 df_union = df1.union(df2)
---> 16 df_union.display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:80, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     76 def df_display(df, *args, **kwargs):
     77     """
     78     df.display() is an alias for display(df). Run help(display) for more information.
     79     """
---> 80     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /

#### Union By Name
- Combines two DataFrames based on column names.
- Column order doesn’t matter — Spark aligns them by name.
- Extra option: allowMissingColumns=True lets you union DataFrames even if one has extra columns (missing values will be filled with null).

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# First DataFrame
data1 = [("A101", 5.2, "Low Fat"),
         ("A102", 8.5, "Regular")]
df1 = spark.createDataFrame(data1, ["Item_Identifier", "Item_Weight", "Item_Fat_Content"])

# Second DataFrame (different column order)
data2 = [(12.0, "A103", "Low Fat"),
         (13.7, "A104", "Regular")]
df2 = spark.createDataFrame(data2, ["Item_Weight", "Item_Identifier", "Item_Fat_Content"])

# Union by name (aligns columns by name, not position)
df_unionByName = df1.unionByName(df2)
display(df_unionByName)


### Date Functions : In PySpark, date functions allow us to parse, format, extract, and perform arithmetic on dates and timestamps.

#### Current_Date

In [0]:
# current_date() generates today’s date (without time) and adds it as a new column.
from pyspark.sql.types import * 
from pyspark.sql.functions import * 

df_CurrDate = df.withColumn('curr_date',current_date())
display(df_CurrDate)

#### date_add

In [0]:
#date_add() in PySpark is used to add a specified number of days to a date column.
Df_week = df_CurrDate.withColumn('Week After', date_add('curr_date', 7))
display(Df_week)
#date_sub() in PySpark is used to subtract a specified number of days from a date column.


####date_sub

In [0]:
#date_sub() in PySpark is used to subtract a specified number of days from a date column.
Df_WeekBefore = Df_week.withColumn('Week Before', date_sub('curr_date', 7))
display(Df_WeekBefore)

#### DateDIFF

In [0]:
#datediff() in PySpark is used to calculate the number of days between two date columns.
Df_Diff = Df_WeekBefore.withColumn('Day Diff', datediff('Week After', 'Week Before'))
display(Df_Diff)
#date_format() in PySpark is used to format a date column into a specified format.
##Df_DateFormat = Df_Diff.withColumn('Week Diff Format',

#### Date Format

In [0]:
#### date_format() in PySpark is used to convert a date or timestamp column into a string with a specific format.
Df_format = Df_Diff.withColumn('curr_date', date_format('curr_date', 'dd-MM-yyyy'))
display(Df_format)